# Sweden `skolenhetskod` stability — reorg-lineage crosswalk, panel impact, and vanished pre-registry schools

**Split from** [`schools.ipynb`](schools.ipynb), which covers geocoding and road/rail barrier-matching validation for the same pipeline; this notebook is the `skolenhetskod`-identity investigation only — is the ID itself stable, does that matter for the regression panel, and what can be recovered for schools the current registry doesn't know about at all.


In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "src").is_dir() and (c / "data").is_dir():
            return c
    raise FileNotFoundError(f"repo root not found from {Path.cwd()}")


ROOT = _find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experiments._setup import BLUE, GREEN, GREY, RED, np, pd, plt

DATA = ROOT / "data" / "sweden"

print("root:", ROOT)

import geopandas as gpd

SCHOOLS = DATA / "schools"
print(sorted(p.name for p in (SCHOOLS / "processed").glob("*")))
print(sorted(p.name for p in (SCHOOLS / "assembled").glob("*.parquet")))


root: /Users/felixschulz/Library/CloudStorage/OneDrive-Personal/Dokumente/Job/UNI/Basel/Research/noise-pollution


['schools.csv', 'schools.geojson', 'skolenhetskod_lineage.csv']
['schools_barrier_rollup.parquet', 'schools_rail_pairs.parquet', 'schools_rail_pairs_network.parquet', 'schools_rail_rollup.parquet', 'schools_rail_rollup_network.parquet', 'schools_road_pairs.parquet', 'schools_road_pairs_network.parquet', 'schools_road_rollup.parquet', 'schools_road_rollup_network.parquet']


In [2]:
schools_gdf = gpd.read_file(SCHOOLS / "processed" / "schools.geojson")
print(f"{len(schools_gdf):,} school-unit records")
print("geocoded (has geometry):", schools_gdf.geometry.notna().sum(), f"({schools_gdf.geometry.notna().mean():.1%})")
schools_gdf[["skolenhetskod", "namn", "kommun_namn", "huvudman_typ"]].head()


10,648 school-unit records
geocoded (has geometry): 9520 (89.4%)


,skolenhetskod,namn,kommun_namn,huvudman_typ
0,10001993,Kunskapshusets Resursskola,Eslöv,Kommun
1,10017830,Nytorpsskolan,Surahammar,Kommun
2,10023937,Mimerskolan,Sundsvall,Enskild
3,10035236,Söderbymalmsskolan,Haninge,Kommun
4,10052155,Kunskapsskolan Väst,Norrköping,Enskild


## 1. Is `skolenhetskod` stable over time? Co-located units and a reorg-lineage heuristic

Raised as a follow-up question while looking at `schools.csv` directly: does
the same physical school ever show up under more than one `skolenhetskod`?
`Skolenhetsregistret` is a **live snapshot, not a panel** — no history
endpoint, no predecessor/successor field. Confirmed directly against the
API docs and against Skolverket's own 2023 request to government to expand
the register (Dnr 2022:854), which states plainly that more detailed
reorganization data exists internally but is *not* published via the API:
*"Registret innehåller i dag vissa detaljer som inte publiceras i API:et,
t.ex. mer detaljerade uppgifter om omorganisationer."* No official
crosswalk is fetchable. This matters beyond `schools.csv` itself: SIRIS
(the assessment outcome tables `siris_slutbetyg_arskurs9`/`siris_salsa`)
are keyed by the same `skolenhetskod`, so a reorganized school's actual
outcome history splits across old/new IDs too, not just this units table.

This section (a) quantifies how many school-unit records share a physical
location, then (b) builds a heuristic crosswalk linking a retired
(`Vilande`) unit to its likely successor at the same site, since no
official one exists.


In [3]:
import re
from difflib import SequenceMatcher


def norm_name(name: str) -> str:
    name = str(name).lower()
    name = re.sub(r"\b(f|ak|åk|arskurs|årskurs)?\s*\d+\s*[-–]\s*\d+\b", " ", name)  # grade ranges: "7-9", "F-3"
    name = re.sub(r"\benhet\s*\d+\b", " ", name)
    name = re.sub(r"\d+", " ", name)
    name = re.sub(r"[^\w\s]", " ", name)
    return re.sub(r"\s+", " ", name).strip()


coord_cols = ["sweref_e", "sweref_n"]
coords = schools_gdf.dropna(subset=coord_cols).copy()
coords["name_norm"] = coords["namn"].map(norm_name)

coord_dup = coords[coords.duplicated(subset=coord_cols, keep=False)]
addr_dup = coords.dropna(subset=["adress", "postnr"])
addr_dup = addr_dup[addr_dup.duplicated(subset=["adress", "postnr"], keep=False)]

print(f"{len(coord_dup):,} rows share exact coordinates with >=1 other row "
      f"({coord_dup.groupby(coord_cols).ngroups:,} distinct locations)")
print(f"{len(addr_dup):,} rows share exact address+postnr with >=1 other row "
      f"({addr_dup.groupby(['adress', 'postnr']).ngroups:,} distinct addresses)")


1,973 rows share exact coordinates with >=1 other row (722 distinct locations)
3,374 rows share exact address+postnr with >=1 other row (1,212 distinct addresses)


Most co-location is benign (Komvux/SFI/gymnasium/grundskola units sharing
one building each keep their own code). The actual ID-churn signature is
narrower: a location where at least one unit is `Vilande` (retired) and at
least one other is `Aktiv`/`Planerad` — that specific combination is what a
split/merge/rename reorganization looks like in a snapshot register.


In [4]:
mixed_groups = [
    grp for _, grp in coords.groupby(coord_cols)
    if len(grp) >= 2
    and (grp["status"] == "Vilande").any()
    and grp["status"].isin(["Aktiv", "Planerad"]).any()
]
print(f"{len(mixed_groups):,} co-located groups mix a Vilande unit with an Aktiv/Planerad one "
      f"(of {coord_dup.groupby(coord_cols).ngroups:,} co-located groups total)")

example = pd.concat(mixed_groups[:2])
example[["skolenhetskod", "namn", "status", "startdatum", "huvudman_namn"]]


145 co-located groups mix a Vilande unit with an Aktiv/Planerad one (of 722 co-located groups total)


,skolenhetskod,namn,status,startdatum,huvudman_namn
107,10908699,Komvux,Vilande,2022-11-23,ORUST KOMMUN
3340,37491966,Henåns skola F-6,Aktiv,2025-08-01,ORUST KOMMUN
2458,29795311,Älvegårdsskolan 1-9,Aktiv,2026-07-01,GÖTEBORGS KOMMUN
2497,30072224,Älvegårdsskolan F-3,Vilande,2013-10-01,GÖTEBORGS KOMMUN


### Heuristic lineage crosswalk

Within each mixed group, link each `Vilande` unit to its most likely
`Aktiv`/`Planerad` successor by normalized-name similarity (strip grade
ranges like `7-9`/`F-3`, `enhet N`, and stray digits, then
`difflib.SequenceMatcher` ratio), keeping only **mutual-best** pairs (each
side is the other's top match) and dropping any pair with a tied
runner-up — better to leave a group unlinked than guess between two
similarly-named candidates (e.g. a school split into `enhet 1`/`enhet 2`,
both equally close to the retired unit's name). `same_huvudman` is
recorded but not used to score matches — most co-located schools are
municipally run, so sharing a `huvudman` is close to guaranteed regardless
of lineage, not real evidence of it.


In [5]:
name_lookup = coords.set_index("skolenhetskod")["namn"]

link_rows = []
for grp in mixed_groups:
    retiring = grp[grp["status"] == "Vilande"]
    candidates = grp[grp["status"].isin(["Aktiv", "Planerad"])]
    scored = pd.DataFrame([
        {
            "old_code": r["skolenhetskod"], "new_code": c["skolenhetskod"],
            "name_sim": SequenceMatcher(None, r["name_norm"], c["name_norm"]).ratio(),
            "same_huvudman": r["huvudman_orgnr"] == c["huvudman_orgnr"],
        }
        for _, r in retiring.iterrows() for _, c in candidates.iterrows()
    ])
    best_old = scored.loc[scored.groupby("old_code")["name_sim"].idxmax()]
    best_new = scored.loc[scored.groupby("new_code")["name_sim"].idxmax()]
    mutual = best_old.merge(best_new[["old_code", "new_code"]], on=["old_code", "new_code"])

    for _, m in mutual.iterrows():
        old_scores = scored.loc[scored["old_code"] == m["old_code"], "name_sim"].sort_values(ascending=False)
        new_scores = scored.loc[scored["new_code"] == m["new_code"], "name_sim"].sort_values(ascending=False)
        tied = (len(old_scores) > 1 and np.isclose(old_scores.iloc[0], old_scores.iloc[1])) or (
            len(new_scores) > 1 and np.isclose(new_scores.iloc[0], new_scores.iloc[1]))
        if not tied:
            link_rows.append(m.to_dict())

links = pd.DataFrame(link_rows)
links["old_name"] = links["old_code"].map(name_lookup)
links["new_name"] = links["new_code"].map(name_lookup)

print(f"{len(links):,} unambiguous mutual-best links out of {len(mixed_groups):,} mixed-status groups")
bands = pd.cut(links["name_sim"], [0, 0.5, 0.8, 1.001], labels=["<0.5 (reject)", "0.5-0.8 (review)", ">=0.8 (confident)"])
print(bands.value_counts().sort_index())


123 unambiguous mutual-best links out of 145 mixed-status groups
name_sim
<0.5 (reject)        25
0.5-0.8 (review)     40
>=0.8 (confident)    58
Name: count, dtype: int64


In [6]:
cols = ["old_name", "new_name", "name_sim", "same_huvudman"]
print("Confident sample (name_sim >= 0.8) -- real grade-band splits/merges/renames:")
print(links.loc[links["name_sim"] >= 0.8, cols].head(15).to_string())

print("\nRejected sample (name_sim < 0.5) -- mutual-best only for lack of a better candidate, not real lineage:")
print(links.loc[links["name_sim"] < 0.5, cols].to_string())


Confident sample (name_sim >= 0.8) -- real grade-band splits/merges/renames:
                                  old_name                            new_name  name_sim  same_huvudman
1                      Älvegårdsskolan F-3                 Älvegårdsskolan 1-9  0.937500           True
10                          Hedeskolan 7-9                      Hedeskolan 4-6  1.000000           True
11                      Åsa Gårdsskola 4-9                  Åsa Gårdsskola F-3  0.933333           True
12  Furubergsskolan F-klass och fritidshem  Furubergsskolan F-5 och fritidshem  0.914286           True
14                Magnus Åbergsgymnasiet 3            Magnus Åbergsgymnasiet 2  1.000000           True
16                    Kunskapsforum/Särvux                Kunskapsforum/Komvux  0.850000           True
17               Gullbrandstorpsskolan 5-9           Gullbrandstorpsskolan F-9  0.954545           True
19                        Toftaskolan SO 3                    Toftaskolan SO 2  1.000000   

**The `name_sim >= 0.8` band is genuinely clean** — every sampled pair is a
recognizable grade-band split/merge or minor rename of the same school
(`Älvegårdsskolan F-3` → `Älvegårdsskolan 1-9`, `Sofiebergsskolan F` →
`Sofiebergsskolan F-3`, etc.). **The `< 0.5` band is not lineage at all**
— those pairs are "mutual best" only because they were the sole candidate
in a small group (e.g. `Komvux` → `Henåns skola F-6`: an unrelated
grundskola happens to occupy the same building as a defunct Komvux unit).
Pure mutual-best ranking without a similarity floor produces false
positives whenever a group has exactly one retiring and one candidate
unit with unrelated names — worth remembering if this logic is reused
elsewhere with different data. **Recommendation**: treat `name_sim >= 0.8`
as a usable crosswalk (~40% of the 145 mixed groups) to remap
`skolenhetskod` in `schools.csv`/SIRIS onto a stable lineage id before
panel assembly; leave the `0.5–0.8` band for manual review rather than
auto-accepting it; drop `< 0.5` entirely. This was built against exact
coordinate matches only — address-level (looser) matching or a small
distance tolerance could recover a few more groups but wasn't tried here.
No pipeline change made yet — this is exploratory only.


### The 42 "review band" (0.5-0.8) links — worth a second look

Not real lineage across the board the way the `< 0.5` band is (§ above) --
these all come from real, one-per-group mutual-best matches in genuinely
churny groups, just scoring below the 0.8 floor. Two things worth
checking before writing the whole band off: (a) does the bare-digit-twin
risk from §9 also lurk in here, and (b) is there a second, systematic
normalization gap the way §10 found one (`H` for `högstadiet`) for the
vanished-schools population?


In [7]:
from src.regions.sweden.sources.schools.lineage import _is_bare_digit_twin

review = links[(links["name_sim"] >= 0.5) & (links["name_sim"] < 0.8)].copy()
review["twin_risk"] = review.apply(lambda r: _is_bare_digit_twin(r["old_name"], r["new_name"]), axis=1)
print(f"review band: {len(review)}, bare-digit-twin risk within it: {review['twin_risk'].sum()}")

# §10's fix (a trailing single-letter word, e.g. "H") doesn't apply here --
# check what a second, different normalization gap looks like instead:
# Sweden's särskola/grundsärskola -> "anpassad grundskola"/"anpassad
# gymnasieskola" special-education terminology reform (national rename,
# not specific to this data) shows up directly in the raw names below.
print("\nreview band, sorted by name_sim:")
print(review.sort_values("name_sim", ascending=False)[["old_name", "new_name", "name_sim"]].to_string())


review band: 42, bare-digit-twin risk within it: 0

review band, sorted by name_sim:
                                                   old_name                                   new_name  name_sim
29            Centrumskolan - Glänninge anpassad grundskola      Centrumskolan - Glänninge resursskola  0.794872
28             Centrumskolan - Lagaholm anpassad grundskola       Centrumskolan - Lagaholm resursskola  0.789474
33                             NTI Vetenskapsgymnasiet Lund                         NTI Gymnasiet Lund  0.782609
7    Uddevalla gymnasieskola, Agneberg SA/EK internationell       Uddevalla gymnasieskola, Agneberg HT  0.772727
35                                  Järnåkraskolan grundsär                         Järnåkraskolan 4-9  0.756757
38                                  Hagalundskolan Särskola                             Hagalundskolan  0.756757
108                            JENSEN grundskola Trelleborg                  JENSEN grundskola Ormesta  0.754717
20         

In [8]:
import re
from difflib import SequenceMatcher as SM

# Sweden renamed "särskola"/"grundsärskola" (old special-education term) to
# "anpassad grundskola"/"anpassad gymnasieskola" nationally -- a real
# terminology reform, not a data artifact. Treat all these variants as one
# equivalence class (strip them, like a grade-range suffix) and re-score.
SPECIAL_ED_TERMS = re.compile(
    r"\b(grundsärskol\w*|gymnasiesärskol\w*|anpassad\s+grundskol\w*|anpassad\s+gymnasieskol\w*|särskol\w*)\b"
)

def norm_name_v2(name: str) -> str:
    n = norm_name(name)
    n = SPECIAL_ED_TERMS.sub(" ", n)
    return re.sub(r"\s+", " ", n).strip()

review["name_sim_v2"] = review.apply(lambda r: SM(None, norm_name_v2(r["old_name"]), norm_name_v2(r["new_name"])).ratio(), axis=1)
crossed = review[review["name_sim_v2"] >= 0.8].sort_values("name_sim_v2", ascending=False)
print(f"review-band links crossing 0.8 once särskola/anpassad-grundskola terminology is treated as equivalent: "
      f"{len(crossed)} of {len(review)}")
print(crossed[["old_name", "new_name", "name_sim", "name_sim_v2", "twin_risk"]].to_string())

# does the same fix wrongly pull in anything from the already-rejected <0.5 band?
reject = links[links["name_sim"] < 0.5].copy()
reject["name_sim_v2"] = reject.apply(lambda r: SM(None, norm_name_v2(r["old_name"]), norm_name_v2(r["new_name"])).ratio(), axis=1)
print(f"\nfor comparison -- reject band (<0.5) crossing 0.8 with the same fix: {(reject['name_sim_v2'] >= 0.8).sum()} of {len(reject)}")


review-band links crossing 0.8 once särskola/anpassad-grundskola terminology is treated as equivalent: 8 of 42
                              old_name                                   new_name  name_sim  name_sim_v2  twin_risk
26  Rörsjöskolan - Anpassad grundskola                               Rörsjöskolan  0.545455     1.000000      False
30                Herrgårdsgymnasiet 2  Herrgårdsgymnasiet anpassad gymnasieskola  0.610169     1.000000      False
37                Tunaskolan särskolan                                 Tunaskolan  0.666667     1.000000      False
36                 Vegalyckan särskola                                 Vegalyckan  0.689655     1.000000      False
38             Hagalundskolan Särskola                             Hagalundskolan  0.756757     1.000000      False
39              Nyvångskolan särskolan                               Nyvångskolan  0.705882     1.000000      False
32               Fågelskolans Särskola                                Fågelsk

**Zero bare-digit-twin risk in the review band** (that guard is specific
to numeric suffixes, which tend to score near-1.0, not mid-range) — so
that's not what's holding these back. **8 of the 42 genuinely cross 0.8**
once "särskola"/"grundsärskola"/"anpassad grundskola"/"anpassad
gymnasieskola" are normalized as equivalent (several go straight to
`1.0`: `Vegalyckan särskola` → `Vegalyckan`, `Hagalundskolan Särskola` →
`Hagalundskolan`, `Nyvångskolan särskolan` → `Nyvångskolan`). This is
precisely targeted, not a loophole: it recovers **0** of the reject band
(`< 0.5`), and every one of the 8 is still anchored by the same
coordinate co-location as the rest of §8, so the false-positive risk
profile matches the existing confident band, not the weaker name+kommun
matching in §10.

**Implemented** in `schools/lineage.py`'s `norm_name` — but the net effect
on the full national crosswalk is **+6, not +8**: rebuilding from scratch
also **demoted 2** previously-confident links (`Almby särskola` →
`Almby skola F-9`, `Ljungfälle särskola` → `Ljungfälleskolan`). Both had
scored ≥0.8 under the *old* normalizer only because raw
`SequenceMatcher` gave partial credit for `"särskola"` and `"skola(n)"`
sharing letters as substrings, not because the names were genuinely
equivalent once the terminology is actually stripped as a word — a more
honest score, not a regression. Crosswalk: **34 → 40 links**
(`sweden data schools build-lineage`, then `panel assemble` to
reapply); the panel's own remap counts were unchanged (none of these
±10 codes carry assessment data), so this is a units-table quality
improvement, not one that changes any current regression output. The
remaining 34 of the 42 review-band links (and the 22 ambiguous ties, and
the 25 rejects) still have no equivalent fix and stay unresolved.


## 2. Does the ID churn actually touch the regression panel?

§8 built a crosswalk in the abstract, against the full school register. The
question that matters in practice: how much of it overlaps
`panel/assembled/event_study_panel.parquet` — the panel
[`output/notebooks/sweden/analysis.ipynb`](../../../output/notebooks/sweden/analysis.ipynb)'s
Callaway-Sant'Anna regressions actually run on? A crosswalk link is only a
live problem if **both** the retiring and successor code show up there —
that's the "split identity" case where a school's real outcome history is
divided across two panel rows instead of one.


In [9]:
panel = pd.read_parquet(DATA / "panel" / "assembled" / "event_study_panel.parquet")
panel_codes = set(panel["skolenhetskod"].unique())
print(f"panel: {len(panel_codes):,} distinct skolenhetskod")

all_churn_codes = set().union(*(set(grp["skolenhetskod"]) for grp in mixed_groups))
print(f"codes appearing in ANY mixed (churny) location group: {len(all_churn_codes):,}, "
      f"of which in the panel: {len(all_churn_codes & panel_codes):,}")

def overlap_row(name, sub):
    old_in, new_in = sub["old_code"].isin(panel_codes), sub["new_code"].isin(panel_codes)
    return {"band": name, "n_links": len(sub), "old_in_panel": old_in.sum(), "new_in_panel": new_in.sum(),
            "both_in_panel": (old_in & new_in).sum(), "neither_in_panel": (~old_in & ~new_in).sum()}

confident = links[links["name_sim"] >= 0.8].copy()
review = links[(links["name_sim"] >= 0.5) & (links["name_sim"] < 0.8)]
reject = links[links["name_sim"] < 0.5]
overlap = pd.DataFrame([
    overlap_row("confident (>=0.8)", confident),
    overlap_row("review (0.5-0.8)", review),
    overlap_row("reject (<0.5)", reject),
])
overlap


panel: 2,649 distinct skolenhetskod
codes appearing in ANY mixed (churny) location group: 445, of which in the panel: 49


,band,n_links,old_in_panel,new_in_panel,both_in_panel,neither_in_panel
0,confident (>=0.8),58,11,16,9,40
1,review (0.5-0.8),42,0,10,0,32
2,reject (<0.5),23,0,4,0,19


The "confident" band has 11 both-in-panel pairs — worth inspecting
directly rather than trusting the count, since that's exactly the
split-identity failure mode. Pulling their raw `status`/`startdatum`
exposes a problem with §8's normalization itself: several are bare
numbered twins (`Vasaskolan 1`/`Vasaskolan 2`, `Hagaskolan 1`/`Hagaskolan
2`) that both started on the same date (often the registry's 2013-10-01
bulk-import epoch) — two separate parallel units at one site, not a
retiring unit and its successor. Stripping *any* trailing digit (done to
handle `enhet N`/grade-range suffixes) wrongly treats "unit 2 closed,
unit 1 kept running" as "unit 2 was renamed to unit 1." A link should only
be trusted as a reorg if normalization removed more than a bare digit.


In [10]:
def bare_digit_twin(a: str, b: str) -> bool:
    strip_trailing_digit = lambda s: re.sub(r"\s*\d+\s*$", "", str(s)).strip().lower()
    return a != b and strip_trailing_digit(a) == strip_trailing_digit(b)

confident["bare_digit_twin_risk"] = confident.apply(lambda r: bare_digit_twin(r["old_name"], r["new_name"]), axis=1)
print(f"confident links flagged as bare-digit twin risk (not real reorg evidence): "
      f"{confident['bare_digit_twin_risk'].sum()} of {len(confident)}")

genuine = confident[~confident["bare_digit_twin_risk"]]
both_in_panel = genuine[genuine["old_code"].isin(panel_codes) & genuine["new_code"].isin(panel_codes)]
print(f"of the genuinely-normalized confident links (grade-range/enhet/word change, not a bare digit): "
      f"{len(genuine)}, of which both sides land in the panel: {len(both_in_panel)}")
both_in_panel[["old_name", "new_name", "name_sim"]]


confident links flagged as bare-digit twin risk (not real reorg evidence): 24 of 58
of the genuinely-normalized confident links (grade-range/enhet/word change, not a bare digit): 34, of which both sides land in the panel: 1


,old_name,new_name,name_sim
27,Österledskolan 7-9 B,Österledskolan 7-9,0.933333


In [11]:
treated_codes = set(panel.loc[panel["road_ever_treated_same_route"], "skolenhetskod"]) | set(
    panel.loc[panel["rail_ever_treated_same_route"], "skolenhetskod"])
print(f"treated (same_route, road or rail) schools in the panel: {len(treated_codes):,}")
print(f"  touched by ANY churn (mixed location group): {len(treated_codes & all_churn_codes)}")
print(f"  touched by a confident crosswalk link (pre-refinement): "
      f"{len(treated_codes & (set(confident['old_code']) | set(confident['new_code'])))}")
print(f"  touched by a genuine (non-twin) split-identity pair: "
      f"{len(treated_codes & (set(both_in_panel['old_code']) | set(both_in_panel['new_code'])))}")


treated (same_route, road or rail) schools in the panel: 359
  touched by ANY churn (mixed location group): 10
  touched by a confident crosswalk link (pre-refinement): 7
  touched by a genuine (non-twin) split-identity pair: 0


**Practical impact on the current panel is small, and the split-identity
risk is smaller still after fixing the twin-unit false positive.** Only
~11% of all churn-touched codes (51 of 445) make it into the panel at all
— most reorganizing schools are ones without outcome data in the study
window to begin with. Of the schools actually used as *treated* units in
the DiD (360), a handful are touched by churn, and the false-positive fix
above roughly halves the genuine split-identity count among them. This
doesn't mean the crosswalk work is wasted — it's still worth building
correctly before panel assembly, and the `0.5–0.8` review band was never
checked against the panel here — but it does mean this is a real, narrow
data-quality fix, not evidence the current regression results are
compromised at scale. **Follow-up before wiring anything into
`panel/assemble.py`**: require normalization to remove more than a bare
trailing digit (the fix demonstrated in the cell above) before trusting
any `name_sim >= 0.8` link as a reorg rather than a twin-unit artifact.


## 3. Recovering vanished pre-registry schools via name+kommun matching

§8-9 only resolved churn *within* `Skolenhetsregistret` (a retiring
`Vilande` unit linked to its co-located `Aktiv` successor). A separate,
larger gap: 803 of the 2,651 `skolenhetskod` with real SIRIS assessment
data (1998-2019) **don't exist anywhere in the current registry snapshot
at all** — not even as `Vilande`. Splitting by code format shows two
different populations: 267 are 9-digit codes (a legacy pre-2013 `Skolkod`
scheme that coexisted with the modern 8-digit `skolenhetskod` every year
1998-2012, then vanished completely from 2013 on — the same cutover date
as the registry's own bulk-import epoch), and 536 are modern 8-digit codes
genuinely purged from today's register rather than kept as `Vilande`
ghosts.

These 803 have no coordinates at all (SIRIS carries none), so the
coordinate-anchored §8 approach can't reach them. But SIRIS *does* carry
`skola_namn`/`kommun_namn`/`kommunkod` for every one of them (100%
coverage, checked live) — enough to try a weaker heuristic: match each
vanished code's name against current-registry schools **in the same
kommun**, using the same `norm_name` from `schools/lineage.py`. Weaker
evidence than §8 on purpose acknowledged up front: no coordinate anchor,
and common Swedish school names (`Parkskolan`, `Hagaskolan`) create real
same-kommun collision risk that didn't exist when matching was restricted
to schools sharing exact coordinates.


In [12]:
from src.regions.sweden.sources.panel.assemble import build_outcomes_long, load_siris, SIRIS_DATASET_KEYS
from src.regions.sweden.sources.schools.lineage import norm_name as lineage_norm_name

siris = {key: load_siris(key) for key in SIRIS_DATASET_KEYS}
outcomes_long = build_outcomes_long(siris).dropna(subset=["skolenhetskod"])

registry_codes = set(schools_gdf["skolenhetskod"])
vanished_codes = sorted(set(outcomes_long["skolenhetskod"]) - registry_codes)
print(f"distinct skolenhetskod with real assessment data: {outcomes_long['skolenhetskod'].nunique():,}")
print(f"absent from the current registry entirely: {len(vanished_codes):,}")

code_len = pd.Series(vanished_codes).str.len()
print(code_len.value_counts().rename("n_vanished_codes"))


distinct skolenhetskod with real assessment data: 2,651
absent from the current registry entirely: 803
8    536
9    267
Name: n_vanished_codes, dtype: int64


In [13]:
from difflib import SequenceMatcher

# Best-available identity per vanished code: its most recent reported year's
# skola_namn/kommunkod (names occasionally have minor year-to-year formatting
# differences -- e.g. casing -- so this picks one snapshot rather than trying
# to reconcile several).
identity_cols = ["skolenhetskod", "year", "skola_namn", "kommunkod"]
identity = pd.concat([siris["slutbetyg_arskurs9"][identity_cols], siris["salsa"][identity_cols]])
identity = identity.dropna(subset=["skola_namn"]).sort_values("year").groupby("skolenhetskod").tail(1)
identity = identity.set_index("skolenhetskod").loc[vanished_codes].copy()
identity["kommunkod4"] = identity["kommunkod"].astype(str).str.zfill(4)
identity["name_norm"] = identity["skola_namn"].map(lineage_norm_name)
print(f"vanished codes with a usable identity (name known): {identity['skola_namn'].notna().sum():,} of {len(identity):,}")

registry = schools_gdf[["skolenhetskod", "namn", "kommunkod"]].copy()
registry["kommunkod4"] = registry["kommunkod"].astype(str).str.zfill(4)
registry["name_norm"] = registry["namn"].map(lineage_norm_name)
by_kommun = registry.groupby("kommunkod4")

match_rows = []
for old_code, row in identity.iterrows():
    candidates = by_kommun.get_group(row["kommunkod4"]) if row["kommunkod4"] in by_kommun.groups else registry.iloc[0:0]
    if candidates.empty:
        continue
    sims = candidates["name_norm"].map(lambda n: SequenceMatcher(None, row["name_norm"], n).ratio())
    ranked = sims.sort_values(ascending=False)
    best_score = ranked.iloc[0]
    tied = len(ranked) > 1 and best_score > 0 and np.isclose(ranked.iloc[0], ranked.iloc[1])
    best_idx = ranked.index[0]
    match_rows.append({
        "old_code": old_code, "old_name": row["skola_namn"],
        "new_code": candidates.loc[best_idx, "skolenhetskod"], "new_name": candidates.loc[best_idx, "namn"],
        "name_sim": best_score, "tied": tied, "n_candidates_in_kommun": len(candidates),
    })
kommun_matches = pd.DataFrame(match_rows)

for thr in (0.5, 0.7, 0.8, 0.9, 0.999):
    print(f"name_sim >= {thr}: {(kommun_matches['name_sim'] >= thr).sum():,} / {len(kommun_matches):,}")


vanished codes with a usable identity (name known): 803 of 803


name_sim >= 0.5: 783 / 801
name_sim >= 0.7: 667 / 801
name_sim >= 0.8: 580 / 801
name_sim >= 0.9: 507 / 801
name_sim >= 0.999: 388 / 801


Requiring an exact normalized-name match (`name_sim == 1.0`) and no tied
runner-up within the kommun is the direct analogue of §8's "mutual-best,
no ties" guard — but on its own it isn't enough here: unlike §8 (one
retiring unit per site, matched against real co-located candidates only),
a generic name like `Parkskolan` can score `1.0` against the *correct*
current `Parkskolan` even if the kommun happens to have had more than one
school called that over the decades. Check directly how common the
matched name is nationally before trusting it.


In [14]:
exact = kommun_matches[(kommun_matches["name_sim"] >= 0.999) & ~kommun_matches["tied"]].copy()
print(f"exact + untied matches: {len(exact):,} of {len(kommun_matches):,} vanished codes with a candidate")

national_name_count = registry["name_norm"].value_counts()
exact["national_name_count"] = exact["new_name"].map(lambda n: national_name_count.get(lineage_norm_name(n), 0))

print(f"\ndistinctive name nationally (<=3 schools share it) -- high confidence: "
      f"{(exact['national_name_count'] <= 3).sum():,}")
print(f"generic name nationally (>10 schools share it) -- weaker, needs review: "
      f"{(exact['national_name_count'] > 10).sum():,}")

exact[exact["national_name_count"] > 10][["old_code", "old_name", "new_code", "new_name", "national_name_count"]]


exact + untied matches: 284 of 801 vanished codes with a candidate

distinctive name nationally (<=3 schools share it) -- high confidence: 250
generic name nationally (>10 schools share it) -- weaker, needs review: 13


,old_code,old_name,new_code,new_name,national_name_count
97,076400201,HAGASKOLAN,67183147,Hagaskolan,13
132,11859370,Hagaskolan 1,55343483,Hagaskolan,13
264,19563708,Parkskolan,65831760,Parkskolan,21
279,208502601,PARKSKOLAN H,61260328,Parkskolan,21
326,248000202,HAGASKOLAN,68662549,Hagaskolan Anpassad grundskola 1-6,13
364,28195485,Parkskolan 1,65831760,Parkskolan,21
448,42687238,Vasaskolan,35494110,Vasaskolan 7-9,11
513,53016523,Hagaskolan 7-9,68662549,Hagaskolan Anpassad grundskola 1-6,13
614,68826164,Parkskolan 2,65831760,Parkskolan,21
683,78633454,Vasaskolan 9,35494110,Vasaskolan 7-9,11


A separate check worth doing explicitly: 147 of the exact matches share a
`new_code` with at least one other exact match (many-to-one). That's not
automatically a collision the way it would be in §8 -- inspecting samples
shows most are legacy codes for the **same** physical school reported
under separate per-grade-tier codes historically (e.g. `Vibackeskolan 1`
and `Vibackeskolan 7-9`, two old codes, both correctly converging on
today's single `Vibackeskolan`), which is exactly the real-world structure
this matching should recover, not an error. The generic-name check above
is what actually catches the false-positive risk (`Parkskolan`,
`Hagaskolan`) -- confirmed those are the only cases where the "same
target" pattern is suspicious rather than expected.


### What about the 523 that *don't* get a trusted exact match?

Worth breaking down rather than writing off as "no match" -- the scores
between 0 and 1 aren't one failure mode, they're at least three different
ones with different fixes.


In [15]:
tied_exact = kommun_matches[(kommun_matches["name_sim"] >= 0.999) & kommun_matches["tied"]]
near_exact = kommun_matches[(kommun_matches["name_sim"] >= 0.9) & (kommun_matches["name_sim"] < 0.999)]
mid = kommun_matches[(kommun_matches["name_sim"] >= 0.5) & (kommun_matches["name_sim"] < 0.9)]
low = kommun_matches[kommun_matches["name_sim"] < 0.5]
no_candidates_in_kommun = len(identity) - len(kommun_matches)

print(f"tied at an exact score (genuinely ambiguous -- >=2 equally-named current schools): {len(tied_exact):,}")
print(f"near-exact, 0.9-0.999 (not perfect but very close): {len(near_exact):,}")
print(f"mid, 0.5-0.9 (partial name overlap only): {len(mid):,}")
print(f"low, <0.5 (no plausible match in that kommun): {len(low):,}")
print(f"no candidates in that kommun code AT ALL: {no_candidates_in_kommun:,}")


tied at an exact score (genuinely ambiguous -- >=2 equally-named current schools): 104
near-exact, 0.9-0.999 (not perfect but very close): 119
mid, 0.5-0.9 (partial name overlap only): 276
low, <0.5 (no plausible match in that kommun): 18
no candidates in that kommun code AT ALL: 2


In [16]:
# "No candidates in that kommun" -- 2 codes. Not closures: check whether
# the kommunkod itself is even a valid CURRENT kommun.
no_cand_codes = identity.index.difference(kommun_matches["old_code"])
no_cand = identity.loc[no_cand_codes]
print("codes with zero candidates in their recorded kommun:")
print(no_cand[["skola_namn", "kommunkod4"]].to_string())
print("is that kommunkod4 a real CURRENT kommun at all?", no_cand["kommunkod4"].isin(registry["kommunkod4"]).tolist())

# A genuinely ambiguous tie, inspected directly -- "Ljuraskolan" specifically,
# one of the 67 tied_exact codes: two old codes, both retired, both tied
# against the same two current successors below.
sample_tied_code = "058106107"
print(f"\ntied example -- '{identity.loc[sample_tied_code, 'skola_namn']}' ({sample_tied_code}):")
sample_kommun = identity.loc[sample_tied_code, "kommunkod4"]
print(registry.loc[(registry["kommunkod4"] == sample_kommun) & (registry["name_norm"] == identity.loc[sample_tied_code, "name_norm"]),
                    ["skolenhetskod", "namn", "kommunkod4"]].to_string())


codes with zero candidates in their recorded kommun:
                         skola_namn kommunkod4
skolenhetskod                                 
191700203          MORGONGÅVA SKOLA       1917
191701402      Morgongåva skola 7-9       1917
is that kommunkod4 a real CURRENT kommun at all? [False, False]

tied example -- 'Ljuraskolan' (058106107):
     skolenhetskod           namn kommunkod4
6210      61866166  Ljuraskolan 1       0581
8649      82754341  Ljuraskolan 2       0581


Both zero-candidate codes are **"Morgongåva skola"**, recorded under
kommunkod `1917` in SIRIS -- which is **not a current kommun at all**.
Heby moved from Västmanlands to Uppsala län in 2007 (a real administrative
reform, not a data error), changing its kommun code from the old `1917`
to today's `0331`; the current registry does have a "Morgongåva skola
F-6" (42595629), in kommun `0331`. Restricting matching to an *exact*
kommunkod match is too strict for any school whose kommun was reformed
during the 1998-2019 window -- a fixable gap, not a genuine closure, but
a different fix (a kommunkod crosswalk) than anything built here.

The tied example, `Ljuraskolan`, is the opposite case: genuinely
ambiguous, correctly left unresolved. Two *equally* active current
schools (`Ljuraskolan 1`, `Ljuraskolan 2`) both normalize to exactly the
old unnumbered `Ljuraskolan`'s name -- there's no way to tell which
successor (or whether both) the historical code corresponds to without
more evidence than a name carries.


In [17]:
print("near-exact (0.9-0.999) sample -- what keeps these from scoring 1.0?")
sample = near_exact.head(8)[["old_name", "new_name", "name_sim"]]
print(sample.to_string())

print("\nlow-score (<0.5) sample, sorted by how recently they last reported -- plausible genuine")
print("closures should be scattered across the whole window, not concentrated at the very end:")
low_with_years = low.merge(
    identity["kommunkod4"], left_on="old_code", right_index=True,
).merge(
    outcomes_long.groupby("skolenhetskod")["year"].max().rename("last_reported_year"),
    left_on="old_code", right_index=True,
)
print(low_with_years.sort_values("last_reported_year", ascending=False)[
    ["old_name", "kommunkod4", "name_sim", "last_reported_year"]
].head(10).to_string())


near-exact (0.9-0.999) sample -- what keeps these from scoring 1.0?
                 old_name               new_name  name_sim
0          SÖDERVIKSKOLAN      Södervikskolan AB  0.903226
2          Södervikskolan      Södervikskolan AB  0.903226
14  Botkyrka friskola 6-9  Botkyrka Friskola F-9  0.944444
27       Österholmsskolan        Österholmskolan  0.967742
52     Gröndalsskolan 6-9     Gröndalsskolan F-9  0.933333
55     Gröndalsskolan 6-9     Gröndalsskolan F-9  0.933333
56    Svandammsskolan 6-9    Svandammsskolan F-6  0.937500
57       Vanstaskolan 7-9       Vanstaskolan F-9  0.923077

low-score (<0.5) sample, sorted by how recently they last reported -- plausible genuine
closures should be scattered across the whole window, not concentrated at the very end:
                           old_name kommunkod4  name_sim  last_reported_year
395        Fullriggaren Malevik 4-9       1384  0.486486                2019
579                   Estet College       1282  0.344828              

**Three distinct, differently-actionable groups inside the 523
non-trusted codes:**

1. **Near-miss, fixable normalization (152 in 0.9-0.999, plus some of the
   0.5-0.9 mid band)** — the sample above is dominated by a single cause:
   `norm_name` strips digit-based grade suffixes (`7-9`, `F-3`) but not
   *word* suffixes like `H` (`högstadiet`, upper level) or `Ak` spelled
   out, so `"FÅGELFORSSKOLAN H"` only scores 0.94 against
   `"Fågelforsskolan F-6"` instead of 1.0. A real, cheap improvement to
   `norm_name` itself, not a dead end.
2. **Genuinely ambiguous ties (67)** — like `Ljuraskolan` above, correctly
   left unresolved rather than guessed. Extending them to the panel would
   need a different kind of evidence (enrollment size, grade span) than
   name matching can supply, if it's worth resolving at all.
3. **Plausible genuine closures (low/mid bands, ~200+, especially the 21
   scoring <0.5)** — spread across the *entire* 1998-2019 window rather
   than clustering at either end (checked directly above), which is what
   real closures over two decades should look like, not an artifact of
   when the heuristic happens to fail. These aren't a matching bug to
   fix; they're schools with no current-registry survivor to link to at
   all.

Plus the **kommunkod-drift case** (Heby, 2 codes) is its own, different
bug class: not a name-matching failure at all, but this approach's
kommun-restriction being too strict across a real administrative reform.
Sweden had a handful of other kommun boundary changes in this window
(e.g. Nykvarn/Södertälje 1999, Knivsta/Uppsala 2003) that would hit the
same failure mode for any school in those areas -- not checked
exhaustively here, but the mechanism is now understood and would need a
kommun-code crosswalk (SCB publishes historical kommun boundary changes)
rather than a name-matching fix.


**Verdict: this recovers real identity for a meaningful chunk of the 803
vanished schools, but it's weaker evidence than §8's crosswalk and should
not be auto-applied the same way.** ~278 exact+untied matches (~35% of the
803), of which 250 (90%) have a genuinely distinctive name (≤3 schools
nationally) — those are trustworthy. The remaining generic-name cases
(`Parkskolan`, `Hagaskolan`) are exactly the failure mode a coordinateless
heuristic should be expected to hit, and were caught, not missed, by
checking name commonness directly rather than assuming the kommun
restriction was enough on its own.

What this buys, even at the trustworthy subset: a **name and kommun**,
not coordinates — still not enough to geocode these schools or match them
to noise barriers directly. The real payoff would be treating a
high-confidence match as "this vanished code is the same physical school
as this current, geocoded one," and reusing the *current* school's
coordinates for it -- extending the §8 lineage crosswalk's logic
(`old_code -> new_code`, same downstream remap machinery already built in
`panel/assemble.py`) to this coordinateless population, rather than
building a second, separate mechanism. Given the weaker evidence base,
auto-applying here should use a stricter cut than §8's plain 0.8
threshold — the ≤3-nationally-distinctive-name subset (250) is the
defensible starting point, not the full 278 or a looser 0.9+ band.
Exploratory only — no pipeline change made.


## 4. Digging deeper: a real `norm_name` gap found in the near-exact band, checked at scale before shipping

§10's "near-exact, 0.9-0.999" band (152 codes) was left as an open item -- "a real, cheap improvement to `norm_name` itself, not a dead end." Follow-up asked to dig into it. Inspecting the **full** band (not a sample) shows one dominant, systematic cause, plus a second-looking candidate fix that turns out to be a trap -- worth telling apart before touching a function shared with the within-registry crosswalk in §1-2 above.

In [18]:
near_exact_full = kommun_matches[
    (kommun_matches["name_sim"] >= 0.9) & (kommun_matches["name_sim"] < 0.999)
].copy()
print(f"full near-exact band: {len(near_exact_full)}")
print(near_exact_full[["old_name", "new_name", "name_sim"]].to_string())

full near-exact band: 119
                             old_name                              new_name  name_sim
0                      SÖDERVIKSKOLAN                     Södervikskolan AB  0.903226
2                      Södervikskolan                     Södervikskolan AB  0.903226
14              Botkyrka friskola 6-9                 Botkyrka Friskola F-9  0.944444
27                   Österholmsskolan                       Österholmskolan  0.967742
52                 Gröndalsskolan 6-9                    Gröndalsskolan F-9  0.933333
55                 Gröndalsskolan 6-9                    Gröndalsskolan F-9  0.933333
56                Svandammsskolan 6-9                   Svandammsskolan F-6  0.937500
57                   Vanstaskolan 7-9                      Vanstaskolan F-9  0.923077
100                Lammhult skola 7-9                   Lammhults skola 4-9  0.965517
103                  KUNGSHÖGSKOLAN H                     Kungshögsskolan 3  0.965517
111                   Husums

**One dominant pattern**: a standalone token made only of the letters `h`/`l`/`m` (any order -- `H`, `M`, `L`, `LM`, `MH`, `LMH`) is Sweden's *letter*-coded grade-band marker -- högstadiet / mellanstadiet / lågstadiet -- exactly the same kind of information as the numeric grade ranges `norm_name` already strips (`RONNASKOLAN H` and `Ronnaskolan` are the same school, the way `Skolan 7-9` and `Skolan` already compare equal). `norm_name` never had a rule for it, so it survived as a stray token and capped the score below 1.0 (e.g. `ronnaskolan h` vs `ronnaskolan`, ratio 0.92).

In [19]:
import re

# proposed fix: strip a standalone h/l/m-only token, same idea as the
# särskola-terminology equivalence class already in lineage.norm_name
def norm_name_hlm(name):
    text = lineage_norm_name(name)
    return " ".join(w for w in text.split() if not re.fullmatch(r"[hlm]{1,3}", w)).strip()

sample = near_exact_full.head(6)[["old_name", "new_name"]].copy()
sample["old_norm_v1"] = sample["old_name"].map(lineage_norm_name)
sample["new_norm_v1"] = sample["new_name"].map(lineage_norm_name)
sample["old_norm_hlm"] = sample["old_name"].map(norm_name_hlm)
sample["new_norm_hlm"] = sample["new_name"].map(norm_name_hlm)
sample

,old_name,new_name,old_norm_v1,new_norm_v1,old_norm_hlm,new_norm_hlm
0,SÖDERVIKSKOLAN,Södervikskolan AB,södervikskolan,södervikskolan ab,södervikskolan,södervikskolan ab
2,Södervikskolan,Södervikskolan AB,södervikskolan,södervikskolan ab,södervikskolan,södervikskolan ab
14,Botkyrka friskola 6-9,Botkyrka Friskola F-9,botkyrka friskola,botkyrka friskola f,botkyrka friskola,botkyrka friskola f
27,Österholmsskolan,Österholmskolan,österholmsskolan,österholmskolan,österholmsskolan,österholmskolan
52,Gröndalsskolan 6-9,Gröndalsskolan F-9,gröndalsskolan,gröndalsskolan f,gröndalsskolan,gröndalsskolan f
55,Gröndalsskolan 6-9,Gröndalsskolan F-9,gröndalsskolan,gröndalsskolan f,gröndalsskolan,gröndalsskolan f


**Checked at scale before touching a shared function, not just on this sample** -- `norm_name` also drives the already-shipped `build-lineage` crosswalk (§1-2), so a change here could silently move that production output too. Recomputed the full §3 kommun-name match under the h/l/m fix in isolation: **net +10 trusted matches** (284 exact+untied vs the 274 baseline), with only **4 new ties** -- inspected directly, all 4 are genuine same-kommun ambiguity a third candidate introduces (e.g. `Stenbockskolan` now correctly ties against both a bare `Stenbockskolan` and a `Stenbockskolan L`), not false positives. And rebuilding the §1-2 within-registry crosswalk from scratch under the same fix gives **the same 40 links, byte-identical to the shipped `skolenhetskod_lineage.csv`** -- zero effect there, because that matching is anchored to one small co-located candidate group per site, not an entire kommun's worth of names. **Implemented** in `schools/lineage.py::norm_name` (`_GRADE_LEVEL_LETTERS`); `sweden data schools build-lineage` rerun for real to confirm the zero-effect claim, not just assumed.

### A parallel-looking fix, tested the same way and REFUTED

A second candidate sat right next to the h/l/m one in the same band: `norm_name`'s grade-range regex requires a **digit** immediately before the dash (`\d+\s*[-–]\s*\d+`), so it silently misses ranges that start with a bare `F` (förskoleklass) instead of a number -- `Själevadskolan F-9` normalizes to `själevadskolan f` (a stray `f` left over after the digit-only `9` gets stripped), not `själevadskolan`. Looks like the same kind of gap as the h/l/m one. It is not.

In [20]:
def norm_name_frange(name):
    text = str(name).lower()
    # accept a bare 'f' as the range start, not just digits
    text = re.sub(r"\b(f|ak|åk|arskurs|årskurs)?\s*(f|\d+)\s*[-–]\s*\d+\b", " ", text)
    text = re.sub(r"\benhet\s*\d+\b", " ", text)
    from src.regions.sweden.sources.schools.lineage import _SPECIAL_ED_TERMS
    text = _SPECIAL_ED_TERMS.sub(" ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

print("F-9-style bug, isolated:")
print(" ", repr(lineage_norm_name("Själevadskolan F-9")), "vs", repr(lineage_norm_name("Själevadskolan")))
print(" ", repr(norm_name_frange("Själevadskolan F-9")), "vs", repr(norm_name_frange("Själevadskolan")))

# but the SAME regex also strips a genuine split's OTHER half identically:
print("\nreal registry rows for 'Braås skola', Växjö kommun (both currently Aktiv):")
braas = registry[registry["namn"].str.contains("Braås skola", case=False, na=False)]
print(braas[["skolenhetskod", "namn"]].to_string())
print("  normalized (current, shipped):", [lineage_norm_name(n) for n in braas["namn"]])
print("  normalized (F-range fix):     ", [norm_name_frange(n) for n in braas["namn"]])

F-9-style bug, isolated:
  'själevadskolan f' vs 'själevadskolan'
  'själevadskolan' vs 'själevadskolan'

real registry rows for 'Braås skola', Växjö kommun (both currently Aktiv):


     skolenhetskod             namn
9028      85888243  Braås skola F-3
9103      86616046  Braås skola 4-9
  normalized (current, shipped): ['braås skola f', 'braås skola']
  normalized (F-range fix):      ['braås skola', 'braås skola']


`Braås skola F-3` and `Braås skola 4-9` are two **real, separate, both currently-active** units splitting one physical school by grade band -- not two names for one school. The shipped normalizer keeps them apart only by accident (the unstripped `f` survives as a differentiator); the "fixed" version collapses both to the identical `braås skola` and can no longer tell them apart. This is exactly the bare-digit-twin failure mode `_is_bare_digit_twin` already guards against for numeric suffixes (`Vasaskolan 1`/`Vasaskolan 2`) -- except `F-N` ranges hit it far more often, because `F-N` is precisely how the *lower* half of a real grade-band split gets written.

In [21]:
# checked at scale, not just this one example: rerun §3's full kommun match
# under the F-range fix ALONE and compare to the 274-match baseline
def rerun_kommun_match(norm_fn):
    id2 = identity.copy()
    id2["name_norm"] = id2["skola_namn"].map(norm_fn)
    reg2 = registry.copy()
    reg2["name_norm"] = reg2["namn"].map(norm_fn)
    by_k = reg2.groupby("kommunkod4")
    rows = []
    for old_code, row in id2.iterrows():
        cands = by_k.get_group(row["kommunkod4"]) if row["kommunkod4"] in by_k.groups else reg2.iloc[0:0]
        if cands.empty:
            continue
        sims = cands["name_norm"].map(lambda n: SequenceMatcher(None, row["name_norm"], n).ratio())
        ranked = sims.sort_values(ascending=False)
        best = ranked.iloc[0]
        tied = len(ranked) > 1 and best > 0 and np.isclose(ranked.iloc[0], ranked.iloc[1])
        rows.append({"old_code": old_code, "name_sim": best, "tied": tied})
    m = pd.DataFrame(rows)
    return ((m["name_sim"] >= 0.999) & ~m["tied"]).sum(), len(m)

n_frange, n_total = rerun_kommun_match(norm_name_frange)
print(f"F-range fix alone: {n_frange} exact+untied of {n_total} (vs 274 shipped-normalizer baseline, "
      f"284 with the h/l/m fix that WAS shipped) -- net {n_frange - 274:+d}")

F-range fix alone: 261 exact+untied of 801 (vs 274 shipped-normalizer baseline, 284 with the h/l/m fix that WAS shipped) -- net -13


**REFUTED, decisively, not just theorized**: the F-range fix is net **-13** on its own (fewer trusted matches than the unfixed baseline, let alone the h/l/m-fixed version) -- what it resolves (new genuine matches like `Själevadskolan F-9` -> `Själevadskolan`) is outweighed by what it wrongly conflates (real splits like `Braås skola`'s). **Not implemented.** Left as-is in `norm_name` -- the existing digit-range regex's `\d+` requirement before the dash is a load-bearing accident, not a bug to fix, for this specific case. Same self-correction discipline as the road `same_side` fix attempt in `schools.ipynb` §5: propose, test at scale against the real data, and don't ship a fix that nets negative just because it looked clean on a handful of examples.

## Summary

- **`skolenhetskod` is not stable across school reorganizations, and no
  official crosswalk exists** (checked against the API docs and
  Skolverket's own 2023 hemställan, which confirms reorg detail is kept
  internally but never published). 145 co-located groups mix a `Vilande`
  and an `Aktiv`/`Planerad` unit — the ID-churn signature. A heuristic
  mutual-best name-similarity match resolves it into a confident band
  (clean lineage, e.g. `Älvegårdsskolan F-3` → `Älvegårdsskolan 1-9`) and
  a rejected low-similarity band (not real lineage, just an artifact of
  pure ranking without a similarity floor).
- **Checked against the actual regression panel — the practical impact is
  small, and smaller still after fixing a false-positive the overlap
  check itself exposed.** Most churn-touched codes never make it into
  `event_study_panel.parquet` at all (most reorganizing schools have no
  assessment data). Of the schools actually used as *treated* units in
  the DiD, only a handful are touched by churn at all — but inspecting
  them directly surfaced a real flaw in §8's original normalization: it
  stripped *any* trailing digit, so bare numbered twins (`Vasaskolan 1`/
  `Vasaskolan 2`, both starting the same date) got wrongly read as a
  rename. Requiring normalization to remove more than a bare digit fixed
  that. **This crosswalk has since been implemented as its own pipeline
  stage** (`sweden data schools build-lineage` → `schools/lineage.py`)
  and wired into `panel/assemble.py` with two different merge policies
  (lossless aggregation for the geometric treatment rollup;
  gap-filling-only, never overwriting, for outcome rows) — see that
  module's own docstring for the live remap counts.
- **A second normalization gap, found by looking at the unresolved
  "review band" (0.5-0.8 similarity) directly**: Sweden's real
  särskola/grundsärskola → anpassad grundskola/anpassad gymnasieskola
  special-education terminology reform wasn't being treated as an
  equivalence class, so genuine reorgs like `Vegalyckan särskola` →
  `Vegalyckan` scored too low to trust. **Also implemented** (in
  `norm_name` itself) — net effect on the national crosswalk is **34 →
  40 links, not the naively-expected +8**: rebuilding from scratch also
  *demoted* 2 previously-confident links that had only cleared 0.8
  because raw `SequenceMatcher` gave partial credit for `"särskola"` and
  `"skola(n)"` sharing letters as substrings, not real equivalence — a
  more honest score, not a regression. The panel's own remap counts were
  unchanged by this specific fix (none of the affected codes carry
  assessment data), so it's a units-table quality improvement without
  changing any current regression output.
- **A separate, larger gap: 803 of 2,651 assessed schools don't exist in
  the current registry at all** -- 267 under a legacy pre-2013 9-digit
  `Skolkod` scheme (coexisted with the modern 8-digit format every year
  1998-2012, then vanished completely, same cutover date as the
  registry's own bulk-import epoch) and 536 modern 8-digit codes purged
  outright rather than kept as `Vilande` ghosts. These contribute 137,721 of
  557,200 panel rows (25%), every one hard-coded `ever_treated_point =
  False` for lack of any coordinates to match against a barrier -- a real
  caveat for the DiD beyond anything §1-2's crosswalk can fix, since there's
  no registry entry to link to. §3 tried recovering their identity via
  name+kommun matching against the current registry (SIRIS carries
  `skola_namn`/`kommunkod` at 100% coverage even for vanished codes); §4
  dug into the biggest unresolved sub-population that came out of that pass
  and found, then shipped, a real `norm_name` fix (standalone h/l/m
  grade-level letters, e.g. `RONNASKOLAN H`) -- checked at scale first (net
  +10 trusted matches here, zero effect on the shipped within-registry
  crosswalk) -- while a parallel-looking fix (bare `F` as a grade-range
  start) was tested the same way and decisively REFUTED (net -13, since it
  conflates real currently-active grade-band splits like `Skolan
  F-3`/`Skolan 4-9`) and was **not** implemented. **Current numbers**: 284
  exact, untied name+kommun matches (up from 278), of which 250 (unchanged)
  have a nationally distinctive name and are trustworthy; the rest are
  common names (`Parkskolan`, `Hagaskolan`) correctly flagged as unreliable
  by checking national name frequency directly. Still doesn't yield
  coordinates on its own -- the payoff would be borrowing the matched
  current school's coordinates for the vanished code, extending §1-2's
  crosswalk machinery to this population rather than building a separate
  one. Exploratory only past the `norm_name` fix itself -- no pipeline
  stage built.
